In [9]:
import warnings
warnings.filterwarnings('ignore')

import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime

# -----------------------------------------------------------------
# 1. 讓使用者動態輸入查詢資訊
# -----------------------------------------------------------------
print("=== 台鐵時刻表爬蟲查詢系統 ===")
print("請輸入車站代碼與名稱 (格式範例: 0930-七堵, 4220-臺南, 1000-臺北)")

start_station = input("請輸入出發站 (預設: 0930-七堵): ").strip()
if not start_station:
    start_station = '0930-七堵'

end_station = input("請輸入目的地站 (預設: 4220-臺南): ").strip()
if not end_station:
    end_station = '4220-臺南'

# 自動推算今天日期作為預設值
today_str = datetime.date.today().strftime('%Y/%m/%d')
ride_date = input(f"請輸入乘車日期 (格式: YYYY/MM/DD, 預設今天 {today_str}): ").strip()
if not ride_date:
    ride_date = today_str

print(f"\n[系統提示] 開始查詢: {ride_date} 從 {start_station} 到 {end_station} 的車次資訊...")

# -----------------------------------------------------------------
# 2. 設定 Request 參數與 Header
# -----------------------------------------------------------------
# 注意：若網頁改版或 CSRF 失效，需手動更換此處的 _csrf 值
payload = {    
    '_csrf': 'e7666daa-56a6-41b9-aba1-34141ed8e04b', 
    'startStation': start_station,
    'endStation': end_station,
    'transfer': 'ONE',
    'rideDate': ride_date,
    'startOrEndTime': 'true',
    'startTime': '00:00',
    'endTime': '23:59',
    'trainTypeList': 'ALL',
    'query': '查詢'
}

headers = { 
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/95.0.4638.54 Safari/537.36'
} 

url_address = 'https://www.railway.gov.tw/tra-tip-web/tip/tip001/tip112/querybytime'

# -----------------------------------------------------------------
# 3. 發送 POST 請求與解析網頁
# -----------------------------------------------------------------
try:
    res_post = requests.post(url_address, data=payload, headers=headers, verify=False) 
    res_post.encoding = 'utf-8' 
    
    soup = BeautifulSoup(res_post.text, 'html.parser')
    
    # 擷取時刻表表格
    tables = soup.select('table[class = "itinerary-controls"]')
    
    if len(tables) == 0:
        print("[錯誤] 找不到任何時刻表資料！可能是 CSRF Token 已過期，或當天無班次，請檢查台鐵網站。")
        exit()
        
    print(f"成功掃出 {len(tables)} 個原始表格片段，開始轉換為資料表...")

    # 使用 pandas 讀取 HTML 表格
    pd_tables = pd.read_html(str(tables))
    target_table = pd_tables[0] # 取得主表格

    # -----------------------------------------------------------------
    # 4. 資料整理與清洗 (比照原程式邏輯)
    # -----------------------------------------------------------------
    column_fields = target_table.columns[:9]
    df1 = pd.DataFrame(target_table, columns=column_fields) 
    df1.index += 1 # 索引從 1 開始

    # 建立空的 DataFrame 用來存放每隔 5 列抽取出來的車次主資料
    df2 = pd.DataFrame(columns=column_fields) 
    jump_rows = 5

    for i in range(1, df1.index.stop, jump_rows):
        tmp = pd.DataFrame(df1.loc[i]).transpose()
        df2 = pd.concat([df2, tmp], ignore_index=True)
        
    df2.index += 1 # 調整新表索引從 1 開始

    # 顯示整理後的結果摘要
    print(f"\n[成功] 總共整理出 {len(df2)} 筆班次資訊！")
    pd.set_option('display.max_columns', 20)
    print(df2.head()) # 預覽前五筆資料

    # -----------------------------------------------------------------
    # 5. 匯出檔案
    # -----------------------------------------------------------------
    # 檔名加入日期與站名避免覆蓋
    file_date = ride_date.replace('/', '')
    file_name_excel = f"台鐵時刻表_{file_date}_{start_station.split('-')[-1]}_到_{end_station.split('-')[-1]}.xlsx"
    file_name_csv = f"台鐵時刻表_{file_date}_{start_station.split('-')[-1]}_到_{end_station.split('-')[-1]}.csv"

    # 匯出成 Excel
    df2.to_excel(file_name_excel, index_label='編號')
    # 匯出成 CSV (使用 utf-8-sig 確保 Excel 打開中文字不亂碼)
    df2.to_csv(file_name_csv, index_label='編號', encoding='utf-8-sig')

    print(f"\n=== 檔案匯出成功 ===")
    print(f"1. Excel 檔已儲存至: {file_name_excel}")
    print(f"2. CSV 檔已儲存至: {file_name_csv}")

except Exception as e:
    print(f"\n[執行出錯] 錯誤訊息: {e}")

=== 台鐵時刻表爬蟲查詢系統 ===
請輸入車站代碼與名稱 (格式範例: 0930-七堵, 4220-臺南, 1000-臺北)


請輸入出發站 (預設: 0930-七堵):  1000-臺北
請輸入目的地站 (預設: 4220-臺南):  4220-臺南
請輸入乘車日期 (格式: YYYY/MM/DD, 預設今天 2026/06/07):  2026/06/07



[系統提示] 開始查詢: 2026/06/07 從 1000-臺北 到 4220-臺南 的車次資訊...
成功掃出 1 個原始表格片段，開始轉換為資料表...

[成功] 總共整理出 27 筆班次資訊！
           車種車次 (始發站 → 終點站)   出發時間   抵達時間       行駛時間  經由 詳細資訊     全票    孩童票  \
1     區間快 1003 ( 七堵 → 新左營 )  05:24  10:14  4 小時 50 分  山線   詳細  $ 574  $ 287   
2  自強(3000) 161 ( 南港 → 臺東 )  06:00  09:52  3 小時 52 分  山線   詳細  $ 891  $ 446   
3        自強 103 ( 汐止 → 潮州 )  06:27  10:58  4 小時 31 分  海線   詳細  $ 891  $ 446   
4        自強 105 ( 基隆 → 潮州 )  07:00  11:31  4 小時 31 分  山線   詳細  $ 891  $ 446   
5       普悠瑪 107 ( 南港 → 潮州 )  07:45  10:52   3 小時 7 分  山線   詳細  $ 891  $ 446   

     敬老票  
1  $ 287  
2  $ 446  
3  $ 446  
4  $ 446  
5  $ 446  

=== 檔案匯出成功 ===
1. Excel 檔已儲存至: 台鐵時刻表_20260607_臺北_到_臺南.xlsx
2. CSV 檔已儲存至: 台鐵時刻表_20260607_臺北_到_臺南.csv
